# Final Predictions: Morgan Fingerprints + Random Forest

This notebook trains the **best model** (Random Forest on Morgan Fingerprints) on the **full training dataset** and makes predictions on the test set.

**Best Model Selection:**
- **Representation**: Morgan Fingerprints (2048-bit)
- **Model**: Random Forest Classifier
- **Performance**: CV AUC = 0.9294, Test AUC = 0.9294

**Steps:**
1. Load full training data (fingerprints + labels)
2. Train Random Forest on ALL training data
3. Load test SMILES strings
4. Generate fingerprints for test molecules
5. Make predictions (activity probabilities)
6. Save predictions to CSV file


### Import Required Libraries


In [3]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.ensemble import RandomForestClassifier

import warnings
warnings.filterwarnings('ignore')

seed = 20231124
np.random.seed(seed)


rf_params = dict(
    random_state=42,
    n_jobs=1,  # single-threaded to keep memory usage low
)


## Step 1: Load Full Training Data

Load the complete training dataset (fingerprints + labels) to train the final model.


In [4]:
# Load training fingerprints and labels
X_train_full = np.load("training_fingerprints_matrix.npy")
y_train_full = np.load("training_fingerprints_labels.npy")

print(f"Training data shape: {X_train_full.shape}")
print(f"Labels shape: {y_train_full.shape}")
print(f"Class balance: {(y_train_full==1).sum()} active / {(y_train_full==0).sum()} inactive")


Training data shape: (202895, 2048)
Labels shape: (202895,)
Class balance: 12514 active / 190381 inactive


## Step 1b: Hold-out test AUC + bootstrap CI
Use a single stratified train/test split (80/20) to mirror the course recipe. Train on the training split with the tuned hyperparameters, predict the entire test split once (point estimate AUC_test), and build a 95% CI via bootstrap resampling of the test set.

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Hold-out evaluation to match instructions
# 1) Split once into train/hold-out (stratified 80/20)
X_train_split, X_test_split, y_train_split, y_test_split = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,
    stratify=y_train_full,
    random_state=seed,
)
print(f"Train split: {X_train_split.shape}, Test split: {X_test_split.shape}")

# 2) Train the tuned model on the train split 
holdout_model = RandomForestClassifier(**rf_params)
holdout_model.fit(X_train_split, y_train_split)

# 3) Get a single AUC point estimate on the entire hold-out set
test_proba_holdout = holdout_model.predict_proba(X_test_split)[:, 1]
test_auc = roc_auc_score(y_test_split, test_proba_holdout)
print(f"Hold-out AUC (point estimate): {test_auc:.4f}")

# 4) Build a 95% CI via bootstrap on the hold-out predictions (percentile method)
B = 100
rng = np.random.default_rng(seed)
n_test = len(y_test_split)
bootstrap_aucs = []
for _ in range(B):
    idx = rng.integers(0, n_test, n_test)  # resample hold-out indices with replacement
    auc_b = roc_auc_score(y_test_split[idx], test_proba_holdout[idx])
    bootstrap_aucs.append(auc_b)

ci_low, ci_high = np.percentile(bootstrap_aucs, [2.5, 97.5])
print(f"95% bootstrap CI for AUC: [{ci_low:.4f}, {ci_high:.4f}]")


Train split: (162316, 2048), Test split: (40579, 2048)
Hold-out AUC (point estimate): 0.9294
95% bootstrap CI for AUC: [0.9236, 0.9352]


## Step 2: Train Final Model on Full Training Data
Train Random Forest classifier on the complete training dataset (after selecting hyperparameters) to generate competition/test-set predictions.

In [6]:
# Train Random Forest on full training data (same settings as tuning)
final_model = RandomForestClassifier(**rf_params)

print("Training Random Forest on full training dataset...")
final_model.fit(X_train_full, y_train_full)
print("Training complete!")


Training Random Forest on full training dataset...
Training complete!


## Step 3: Load Test Data and Generate Fingerprints

Load test SMILES strings and generate Morgan fingerprints using the same parameters as training.


In [7]:
# Load test SMILES
test_data = pd.read_csv("test_smiles.csv")
print(f"Test data shape: {test_data.shape}")
print(f"Test columns: {test_data.columns.tolist()}")
test_data.head()


Test data shape: (67631, 2)
Test columns: ['INDEX', 'SMILES']


,INDEX,SMILES
0,202896,O=C(N/N=C/c1ccc(Br)s1)c1ccco1
1,202897,Cc1nc(SCC(=O)Nc2c(C)n(C)n(-c3ccccc3)c2=O)n[nH]1
2,202898,O=C(NC(=S)NCc1ccccc1)C1CC1
3,202899,C/C=C/C(=O)NCCc1ccc(OC)c(OC)c1
4,202900,Cc1cc(C)c(OCC(=O)N/N=C/c2ccc3c(c2)OCO3)c(C)c1


In [8]:
# Generate fingerprints for test molecules (same as training: radius=2, nBits=2048)
def generate_fingerprint(smiles):
    """Generate Morgan fingerprint for a SMILES string"""
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)

print("Generating fingerprints for test molecules...")
test_fingerprints = []
invalid_count = 0

for idx, smiles in enumerate(test_data["SMILES"]):
    fp = generate_fingerprint(smiles)
    if fp is None:
        invalid_count += 1
        # Use zero vector for invalid molecules
        fp = np.zeros(2048, dtype=int)
    else:
        fp = np.array(fp)
    test_fingerprints.append(fp)
    
    if (idx + 1) % 10000 == 0:
        print(f"  Processed {idx + 1}/{len(test_data)} molecules...")

X_test = np.array(test_fingerprints)
print(f"\nTest fingerprints shape: {X_test.shape}")
print(f"Invalid molecules: {invalid_count}")


Generating fingerprints for test molecules...


[16:55:54] DEPRECATION WARNING: please use MorganGenerator
[16:55:54] DEPRECATION WARNING: please use MorganGenerator
[16:55:54] DEPRECATION WARNING: please use MorganGenerator
[16:55:54] DEPRECATION WARNING: please use MorganGenerator
[16:55:54] DEPRECATION WARNING: please use MorganGenerator
[16:55:54] DEPRECATION WARNING: please use MorganGenerator
[16:55:54] DEPRECATION WARNING: please use MorganGenerator
[16:55:54] DEPRECATION WARNING: please use MorganGenerator
[16:55:54] DEPRECATION WARNING: please use MorganGenerator
[16:55:54] DEPRECATION WARNING: please use MorganGenerator
[16:55:54] DEPRECATION WARNING: please use MorganGenerator
[16:55:54] DEPRECATION WARNING: please use MorganGenerator
[16:55:54] DEPRECATION WARNING: please use MorganGenerator
[16:55:54] DEPRECATION WARNING: please use MorganGenerator
[16:55:54] DEPRECATION WARNING: please use MorganGenerator
[16:55:54] DEPRECATION WARNING: please use MorganGenerator
[16:55:54] DEPRECATION WARNING: please use MorganGenerat

  Processed 10000/67631 molecules...


[16:56:05] DEPRECATION WARNING: please use MorganGenerator
[16:56:05] DEPRECATION WARNING: please use MorganGenerator
[16:56:05] DEPRECATION WARNING: please use MorganGenerator
[16:56:05] DEPRECATION WARNING: please use MorganGenerator
[16:56:05] DEPRECATION WARNING: please use MorganGenerator
[16:56:05] DEPRECATION WARNING: please use MorganGenerator
[16:56:05] DEPRECATION WARNING: please use MorganGenerator
[16:56:05] DEPRECATION WARNING: please use MorganGenerator
[16:56:05] DEPRECATION WARNING: please use MorganGenerator
[16:56:05] DEPRECATION WARNING: please use MorganGenerator
[16:56:05] DEPRECATION WARNING: please use MorganGenerator
[16:56:05] DEPRECATION WARNING: please use MorganGenerator
[16:56:05] DEPRECATION WARNING: please use MorganGenerator
[16:56:05] DEPRECATION WARNING: please use MorganGenerator
[16:56:05] DEPRECATION WARNING: please use MorganGenerator
[16:56:05] DEPRECATION WARNING: please use MorganGenerator
[16:56:05] DEPRECATION WARNING: please use MorganGenerat

  Processed 20000/67631 molecules...


...

  Processed 50000/67631 molecules...


[16:56:50] DEPRECATION WARNING: please use MorganGenerator
[16:56:50] DEPRECATION WARNING: please use MorganGenerator
[16:56:50] DEPRECATION WARNING: please use MorganGenerator
[16:56:50] DEPRECATION WARNING: please use MorganGenerator
[16:56:50] DEPRECATION WARNING: please use MorganGenerator
[16:56:50] DEPRECATION WARNING: please use MorganGenerator
[16:56:50] DEPRECATION WARNING: please use MorganGenerator
[16:56:50] DEPRECATION WARNING: please use MorganGenerator
[16:56:50] DEPRECATION WARNING: please use MorganGenerator
[16:56:50] DEPRECATION WARNING: please use MorganGenerator
[16:56:50] DEPRECATION WARNING: please use MorganGenerator
[16:56:50] DEPRECATION WARNING: please use MorganGenerator
[16:56:50] DEPRECATION WARNING: please use MorganGenerator
[16:56:50] DEPRECATION WARNING: please use MorganGenerator
[16:56:50] DEPRECATION WARNING: please use MorganGenerator
[16:56:50] DEPRECATION WARNING: please use MorganGenerator
[16:56:50] DEPRECATION WARNING: please use MorganGenerat

  Processed 60000/67631 molecules...


[16:57:01] DEPRECATION WARNING: please use MorganGenerator
[16:57:01] DEPRECATION WARNING: please use MorganGenerator
[16:57:01] DEPRECATION WARNING: please use MorganGenerator
[16:57:01] DEPRECATION WARNING: please use MorganGenerator
[16:57:01] DEPRECATION WARNING: please use MorganGenerator
[16:57:01] DEPRECATION WARNING: please use MorganGenerator
[16:57:01] DEPRECATION WARNING: please use MorganGenerator
[16:57:01] DEPRECATION WARNING: please use MorganGenerator
[16:57:01] DEPRECATION WARNING: please use MorganGenerator
[16:57:01] DEPRECATION WARNING: please use MorganGenerator
[16:57:01] DEPRECATION WARNING: please use MorganGenerator
[16:57:01] DEPRECATION WARNING: please use MorganGenerator
[16:57:01] DEPRECATION WARNING: please use MorganGenerator
[16:57:01] DEPRECATION WARNING: please use MorganGenerator
[16:57:01] DEPRECATION WARNING: please use MorganGenerator
[16:57:01] DEPRECATION WARNING: please use MorganGenerator
[16:57:01] DEPRECATION WARNING: please use MorganGenerat


Test fingerprints shape: (67631, 2048)
Invalid molecules: 0


## Step 4: Make Predictions

Use the trained model to predict activity probabilities for test molecules.


In [9]:
# Predict activity probabilities (probability of being active)
print("Making predictions...")
test_proba = final_model.predict_proba(X_test)[:, 1]  # Probability of class 1 (active)

print(f"Predictions shape: {test_proba.shape}")
print(f"Prediction range: [{test_proba.min():.4f}, {test_proba.max():.4f}]")
print(f"Mean prediction: {test_proba.mean():.4f}")


Making predictions...
Predictions shape: (67631,)
Prediction range: [0.0000, 1.0000]
Mean prediction: 0.0583


## Step 5: Save Predictions

Save predictions to a CSV file with INDEX and predicted activity probability.


In [10]:
# Create output dataframe
predictions_df = pd.DataFrame({
    'INDEX': test_data['INDEX'],
    'ACTIVE': test_proba
})

# Save to CSV
output_file = 'test_predictions.csv'
predictions_df.to_csv(output_file, index=False)
print(f"Predictions saved to '{output_file}'")
print(f"\nFirst few predictions:")
print(predictions_df.head(10))
print(f"\nSummary statistics:")
print(predictions_df['ACTIVE'].describe())


Predictions saved to 'test_predictions.csv'

First few predictions:
    INDEX  ACTIVE
0  202896    0.69
1  202897    0.00
2  202898    0.85
3  202899    0.00
4  202900    0.02
5  202901    0.15
6  202902    0.00
7  202903    0.00
8  202904    0.01
9  202905    0.00

Summary statistics:
count    67631.000000
mean         0.058264
std          0.151684
min          0.000000
25%          0.000000
50%          0.010000
75%          0.030000
max          1.000000
Name: ACTIVE, dtype: float64


## Step 6: Create Submission File

Create the submission file in the required format:
- First row: Estimated AUC of the chosen model
- Remaining rows: Predicted probabilities (one per line, no header)


In [11]:

group_no = "14"  

# Estimated AUC from cross-validation (from notebook 2b)
# This is the CV AUC of the best model (Random Forest on Fingerprints)
estimated_auc = 0.9294  # Update if you have a more precise value

# Create submission file content
# First row: AUC estimate
# Remaining rows: predictions in same order as test file
submission_data = [estimated_auc] + test_proba.tolist()

# Save to .txt file (one value per line, no header)
submission_file = f"{group_no}.txt"
with open(submission_file, 'w') as f:
    for value in submission_data:
        f.write(f"{value}\n")

print(f"Submission file saved as '{submission_file}'")
print(f"File contains {len(submission_data)} rows (1 AUC + {len(test_proba)} predictions)")
print(f"\nFirst 5 lines of file:")
with open(submission_file, 'r') as f:
    for i, line in enumerate(f):
        if i < 5:
            print(f"  {line.strip()}")
        else:
            break


Submission file saved as '14.txt'
File contains 67632 rows (1 AUC + 67631 predictions)

First 5 lines of file:
  0.9294
  0.69
  0.0
  0.85
  0.0


## Step 7: Validate Submission File

Check that the submission file is formatted correctly.

In [12]:
# Validate the submission file
submission_df = pd.read_csv(submission_file, header=None)

print(f"File shape: {submission_df.shape}")
print(f"Expected shape: (67632, 1)")
print(f"Shape matches: {submission_df.shape == (67632, 1)}")

# Check that all values are between 0 and 1
all_valid = np.all((submission_df.values >= 0) & (submission_df.values <= 1))
print(f"\nAll values in [0, 1]: {all_valid}")

# Check first value (should be AUC)
auc_value = submission_df.iloc[0, 0]
print(f"\nFirst row (AUC estimate): {auc_value:.4f}")

# Check prediction statistics
predictions_only = submission_df.iloc[1:, 0].values
print(f"\nPrediction statistics:")
print(f"  Min: {predictions_only.min():.6f}")
print(f"  Max: {predictions_only.max():.6f}")
print(f"  Mean: {predictions_only.mean():.6f}")


File shape: (67632, 1)
Expected shape: (67632, 1)
Shape matches: True

All values in [0, 1]: True

First row (AUC estimate): 0.9294

Prediction statistics:
  Min: 0.000000
  Max: 1.000000
  Mean: 0.058264
